Paper Pipeline Protoyping

In [175]:
# psql -h localhost -d research_platform -U postgres -f platform_schema.sql

In [176]:
import torch
print(f"PyTorch version: {torch.__version__}")
torch.cuda.is_available()

PyTorch version: 2.7.0


True

In [177]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_postgres import PGVector
import pandas as pd
from sqlalchemy import create_engine
import json
import psycopg
import ray
import dask.dataframe as dd
import dask
import modin.pandas as mdf
import os
import polars as pl
from dotenv import load_dotenv
load_dotenv()
ray.init(num_gpus=1, ignore_reinit_error=True)

2025-06-09 18:57:26,063	INFO worker.py:1718 -- Calling ray.init() again after it has already been called.


Python version:,3.12.10
Ray version:,2.46.0


In [178]:
# list the available CPU cores
total_cores = ray.cluster_resources()['CPU']
print(f"Total CPU cores available: {total_cores}")
ray.available_resources()


Total CPU cores available: 128.0


{'node:192.168.50.241': 1.0,
 'accelerator_type:G': 1.0,
 'node:__internal_head__': 1.0,
 'CPU': 128.0,
 'memory': 67306479207.0,
 'object_store_memory': 28845604587.0,
 'GPU': 1.0}

In [179]:
def create_embedding(model, sentences):
  """
  Create embeddings for a list of sentences using the specified model.
  
  Args:
      model: The model to use for creating embeddings.
      sentences: A list of sentences to embed.
  
  Returns:
      A list of embeddings for the input sentences.
  """
  embeddings = model.encode(sentences, convert_to_tensor=True)
  return embeddings

In [180]:
arxiv_bert_model = SentenceTransformer('ArtifactAI/arxiv-distilroberta-base-GenQ', device='cuda' if torch.cuda.is_available() else 'cpu')
# output_embeddings = create_embedding(
#   model=arix_bert_model,
#   sentences=["This is a test sentence", "This is another test sentence"]
# )

In [181]:
@ray.remote(num_gpus=1)
def process_paper_metadata_embeddings(model,dataframe):
  """
  Process a JSON file containing paper metadata and extract relevant information.
  
  """
  dataframe["embedding"] = dataframe.apply(
    lambda row: create_embedding(
      model,
      row['abstract'],
    ).tolist(),
    axis=1
  )
  dataframe = dataframe.rename(columns={'abstract': 'context'})
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe[["paper_id", "embedding", "context"]]
  
  return dataframe


In [182]:
async def store_row_in_postgres(connection, row):
  """
  Store a row of data in the PostgreSQL database.
  
  Args:
      connection: The database connection object.
      row: The row of data to store.
  
  """
  paper_id = row['paper_id']
  embedding = row['embedding']
  context = row['context']
  
  # Insert the data into the database
  await connection.execute(
    "INSERT INTO paper_vector_db (paper_id, embedding, context) VALUES (%s, %s, %s)",
    (paper_id, embedding, context),
  )
  await connection.commit()

In [183]:
connection = await psycopg.AsyncConnection.connect(
  os.environ['SELF_HOSTED_POSTGRES_CONNECTION']
)

### Parallelizing paper upload



Use Modin Dataframes for parallel processing, wraps pandas in ray context


In [184]:
def fetch_computer_science_paper_metadata(dataframe):
  # 
  computer_science_paper_metadata_mdf = dataframe[
    dataframe['categories'].fillna('').str.contains('cs.', case=False, na=False)
  ]
  return computer_science_paper_metadata_mdf
  

In [185]:
def process_paper_metadata_papers(dataframe):
#  Index(['paper_id', 'submitter', 'authors', 'title', 'comments', 'journal-ref',
#        'doi', 'report-no', 'categories', 'license', 'abstract', 'versions',
#        'update_date', 'authors_parsed'],
#       dtype='object')
  print("Initial keys",dataframe.keys())
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe.rename(columns={'report-no': 'report_number'})
  dataframe = dataframe.fillna('').astype(str)
  print("Output keys",dataframe.keys())
  dataframe = dataframe[["paper_id", "title", "abstract", "doi", "report_number"]]
  return dataframe 
  

In [186]:
def process_paper_document_metadata(dataframe):
  
  dataframe = dataframe.rename(columns={'id': 'paper_id'})
  dataframe = dataframe.rename(columns={'title': 'file_name'})
  dataframe["owner_id"] = "public"
  dataframe["group_id"] = "public"
  dataframe["url"] = dataframe.apply(lambda row: f"/arxiv.org/pdf/{row['paper_id']}", axis=1)
  print(dataframe.keys())
  return dataframe[["paper_id", "owner_id", "group_id", "file_name", "url"]]



In [188]:
paper_metadata_df = pd.read_json('arxiv-metadata-oai-snapshot.json', lines=True).head(10)

In [190]:
display(paper_metadata_df.head())

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [191]:
engine = create_engine(os.environ["SELF_HOSTED_POSTGRES_CONNECTION"])
alchemy_connection = engine.connect()

In [192]:
cs_paper_df = fetch_computer_science_paper_metadata(paper_metadata_df)
cs_paper_mdf = mdf.DataFrame(cs_paper_df)


paper vector db upload


In [193]:
embedding_mdf = ray.get(process_paper_metadata_embeddings.remote(arxiv_bert_model, paper_metadata_df))
embedding_mdf.to_sql("paper_vector_db", con=alchemy_connection, if_exists="replace", index=False)

10

In [194]:
process_paper_metadata_papers(cs_paper_df.copy()).to_sql('paper_metadata',alchemy_connection, if_exists='replace', index=False)

Initial keys Index(['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi',
       'report-no', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')
Output keys Index(['paper_id', 'submitter', 'authors', 'title', 'comments', 'journal-ref',
       'doi', 'report_number', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')


2

In [195]:
display(cs_paper_mdf.keys())

Index(['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi',
       'report-no', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')

In [196]:
process_paper_document_metadata(cs_paper_df.copy()).to_sql("documents", con=alchemy_connection, if_exists="replace", index=False)

Index(['paper_id', 'submitter', 'authors', 'file_name', 'comments',
       'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract',
       'versions', 'update_date', 'authors_parsed', 'owner_id', 'group_id',
       'url'],
      dtype='object')


2

In [197]:
display(cs_paper_mdf.head())

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"


In [198]:
ray.shutdown()

I0000 00:00:1749520816.364555  100301 chttp2_transport.cc:1182] ipv4:192.168.50.241:58119: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {created_time:"2025-06-09T19:00:16.364551722-07:00", http2_error:2, grpc_status:14}
